In [1]:
import pandas as pd, numpy as np, os
df = pd.read_csv("../data/processed/cleaned_dataset.csv")

# Salary relative to seniority peers, so experience doesn't confound the comparison
df["log_salary"] = np.log1p(df["estimated_annual_salary_lpa"])
df["sal_resid"] = df["log_salary"] - df.groupby("seniority_level")["log_salary"].transform("mean")

for c in ["company_size", "employment_type", "work_mode", "open_to_work",
          "uses_generative_ai_tools", "connections_tier"]:
    print(df.groupby(c)[["employability_score", "sal_resid"]].mean().round(2), "\n")

                        employability_score  sal_resid
company_size                                          
Enterprise/MNC (5000+)                48.75       0.00
Large (501-5000)                      48.70      -0.00
Small-Mid (51-500)                    49.73      -0.01
Startup (1-50)                        49.53       0.00 

                 employability_score  sal_resid
employment_type                                
Contract                       49.30      -0.01
Freelance/Gig                  49.32       0.00
Full-time                      49.13       0.00
Internship                     50.06       0.01 

           employability_score  sal_resid
work_mode                                
Hybrid                   51.50        0.0
On-site                  46.55       -0.0
Remote                   51.04       -0.0 

              employability_score  sal_resid
open_to_work                                
No                          49.55      -0.00
Yes                         48.

In [2]:
sen_order = ["Entry-level", "Associate", "Mid-Senior", "Senior", "Lead/Principal", "Director+"]
edu_order = ["Bootcamp/Certification-only", "Diploma", "Bachelor's", "Master's", "PhD"]
upd_order = ["6+ months ago", "3-6 months ago", "1-3 months ago", "This month", "This week"]
con_order = ["<50", "50-150", "150-500", "500+", "1000+"]

df["seniority_ord"] = df["seniority_level"].map({v: i for i, v in enumerate(sen_order)})
df["education_ord"] = df["education_level"].map({v: i for i, v in enumerate(edu_order)})
df["update_ord"] = df["last_profile_update"].map({v: i for i, v in enumerate(upd_order)})
df["connections_ord"] = df["connections_tier"].map({v: i for i, v in enumerate(con_order)})
for c in ["in_demand_skill_flag", "open_to_work", "uses_generative_ai_tools"]:
    df[c + "_bin"] = (df[c] == "Yes").astype(int)
df["log_endorsements"] = np.log1p(df["endorsement_count"])

# Targets
df["high_employability"] = (df["employability_score"] >= df["employability_score"].median()).astype(int)
print(df["high_employability"].value_counts(normalize=True).round(3))
print(df[["seniority_ord", "education_ord", "update_ord", "connections_ord"]].isna().sum())

high_employability
1    0.501
0    0.499
Name: proportion, dtype: float64
seniority_ord      0
education_ord      0
update_ord         0
connections_ord    0
dtype: int64


In [3]:
df.to_csv("../data/processed/features_dataset.csv", index=False)
print(df.shape)

(12000, 37)
